# Проект спринта "Секреты Темнолесья"

- Автор: Виктория Саватнеева
- Дата: 02.08.2025

### Цели и задачи проекта

<font color='#777778'>Целью проекта является знакомство с данными, их предобработка и дальнейшая обработка. В задачи входит анализ на наличие пропусков, выявление явных/неявных дубликатов, приведение значений к нужному типу данных, а также задачами является категоризация обработанных данных и выделение топа платформ по количеству игр. </font>

### Описание данных
<font color='#777778'>
    
- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию. </font>

### Содержимое проекта

<font color='#777778'>

1. Загрузка данных и знакомство с ними
2. Проверка ошибок в данных и их предобработка
3. Фильтрация данных
4. Категоризация данных
5. Итоговый вывод
    
    </font>

## 1. Загрузка данных и знакомство с ними

In [1]:
import pandas as pd

Данные представлены 11 столбцами и всего 16955 строками, нумерация начинается с 0. Данные соответствуют описанию, но в данных встречаются пропуски, причем в некоторых столбцах значительные (например, 8 столбец оценка критиков содержит наибольшее количество пропусков в данных). Не везде используются верные типы данных: например, в столбцах, где ожидается числовое значение, тип object.

Столбцы расположены в хаотичном порядке, продажи по странам я бы перенесла в число последних столбцов, а рейтинг, оценку пользователей и критиков поместила бы в начало.

Названия столбцов также следует переименовать на более корректные названия, удобные для анализа и отражающие суть содержимых данных. Для корректности названия лучше использовать единый стиль написания (na_sales, eu_sales, jp_sales)

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [3]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

### 2.2. Типы данных

In [5]:
print(df)

                                name platform  year_of_release         genre  \
0                         Wii Sports      Wii           2006.0        Sports   
1                  Super Mario Bros.      NES           1985.0      Platform   
2                     Mario Kart Wii      Wii           2008.0        Racing   
3                  Wii Sports Resort      Wii           2009.0        Sports   
4           Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing   
...                              ...      ...              ...           ...   
16951  Samurai Warriors: Sanada Maru      PS3           2016.0        Action   
16952               LMA Manager 2007     X360           2006.0        Sports   
16953        Haitaka no Psychedelica      PSV           2016.0     Adventure   
16954               Spirits & Spells      GBA           2003.0      Platform   
16955            Winning Post 8 2016      PSV           2016.0    Simulation   

       na_sales eu_sales jp_sales  othe

Видим, что некорректно указан тип данных в столбце year_of_release, скорее всего данные вносились с целочисленным форматом ячейки, из-за чего могла возникнуть такая ошибка.

Заменим тип данных только в столбцах с некорректным типом, но корректной записью данных, в которых не встречаются пропуски

In [6]:
import numpy as np

In [7]:
df[['eu_sales', 'jp_sales']] = df[['eu_sales', 'jp_sales']].replace('unknown', np.nan).astype('float64')

In [8]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB
None


### 2.3. Наличие пропусков в данных

In [9]:
na_per_column = df.isna().sum()
total_rows = len(df)
relative_na = (df.isna().sum() / total_rows) * 100
print(na_per_column, relative_na)

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         6804
rating             6871
dtype: int64 name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         40.127389
rating             40.522529
dtype: float64


Основное количество пропусков приходится на такие столбцы как: critic_score, user_score, rating. Предположим, что пропуски могли возникнуть из-за отсутствия анализа и вследствие заполнения игроками или критиками соответствующих оценок. Здесь количество пропусков исчисляется тысячами: 8714, 6804 и 6871 соответственно. 
Меньшее количество пропусков приходится на год выпуска игры - 275, могли возникнуть из-за человеческого фактора или отсутствия определенной официальной даты выхода игры.
Пропуски в других столбцах меньше 10 и, скорее всего, возникли из-за человеческого фактора или отсутствия данных.

В столбцах, где пропуски исчисляются тысячами, можно оставить, либо заменить на усредненные значения по аналогичным жанрам игр

В столбце года выпуска игры и других, где пропусков меньше 10, пропуски можно удалить, так как на анализ это не должно повлиять

В столбце rating удалим пропуски, так как значения, представленные в строках, не числовые. В столбце year_of_release, genre удалим пропуски из-за отсутствия возможности их заменить, а в столбцах о продажах удалим пропуски за ненадобностью:

In [10]:
df = df.dropna(subset = ['rating', 'year_of_release', 'genre', 'eu_sales', 'jp_sales'])

Столбец user_score приведем к числовому и заменим пропуски на значение-индикатор:

In [11]:
df['user_score'].unique()

array(['8', '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4', '8.2',
       '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2', '8.9',
       '6.4', '7.8', '7.5', '2.6', '7.2', '9.2', '7', '7.3', '4.3', '7.6',
       '5.7', '5', '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8', '6.1',
       '6.7', '5.4', nan, '4', '4.9', '4.5', '9.3', '6.2', '4.2', '6',
       '3.7', '4.1', '5.8', '5.6', '5.5', '4.4', '4.6', '5.9', '3.9',
       '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5', '2.5', '1.9', '3',
       '2.7', '2.2', '2', '9.5', '2.1', '3.6', '2.8', '1.8', '3.8', '0',
       '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
       '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9', '1', '9.7'],
      dtype=object)

Значение "tbd" заменим на NaN и приведем столбец к типу float64, затем вычислим среднее значение по жанру и заменим пропуски

In [12]:
df.loc[:, 'user_score'] = df['user_score'].replace('tbd', np.nan)

In [13]:
df.loc[:, 'user_score'] = df['user_score'].astype('float64')

In [14]:
print(df.groupby('genre')['user_score'].mean())

genre
ACTION          6.571429
Action          7.054204
Adventure       7.125442
FIGHTING        6.500000
Fighting        7.293671
MISC                 NaN
Misc            6.811778
PLATFORM        6.500000
PUZZLE          4.200000
Platform        7.306840
Puzzle          7.140476
RACING          5.166667
ROLE-PLAYING    8.533333
Racing          7.028776
Role-Playing    7.618996
SHOOTER         6.840000
SIMULATION      3.300000
SPORTS          6.050000
STRATEGY             NaN
Shooter         7.051592
Simulation      7.147181
Sports          6.967560
Strategy        7.322074
Name: user_score, dtype: float64


Видим, что пропуски актуальны только для двух строк из-за возникших дубликатов, их обработаем в следующем разделе, а здесь пропуски оставим

В столбце crititc_score заменим пропуски на значение-индикатор:

Вычислим среднее значение для каждого жанра:

In [15]:
print(df.groupby('genre')['critic_score'].mean())

genre
ACTION          64.857143
Action          66.632808
Adventure       65.013072
FIGHTING        67.000000
Fighting        69.125926
MISC            49.000000
Misc            66.657588
PLATFORM        59.500000
PUZZLE          73.000000
Platform        67.985772
Puzzle          67.191781
RACING          66.750000
ROLE-PLAYING    71.333333
Racing          67.924451
Role-Playing    72.615595
SHOOTER         72.250000
SIMULATION      33.000000
SPORTS          62.000000
STRATEGY              NaN
Shooter         70.177229
Simulation      68.706052
Sports          72.004299
Strategy        72.359862
Name: critic_score, dtype: float64


Видим, что пропуски актуальны только для одной строки из-за возникших дубликатов, их обработаем в следующем разделе, а здесь пропуски оставим

### 2.4. Явные и неявные дубликаты в данных

Из предыдушего раздела видим, что в столбце genre присутствуют дубликаты из-за разного написания, приведем все значения строк к нижнему регистру:

In [16]:
df['genre'] = df['genre'].str.lower()

In [17]:
print(df['genre'].unique())

['sports' 'racing' 'platform' 'misc' 'action' 'puzzle' 'shooter'
 'fighting' 'simulation' 'role-playing' 'adventure' 'strategy']


Проверим столбец name на дубликаты:

In [18]:
print(df['name'].unique())

['Wii Sports' 'Mario Kart Wii' 'Wii Sports Resort' ... 'Mega Brain Boost'
 'STORM: Frontline Nation' 'Plushees']


Проверим столбец Platform на дубликаты:

In [19]:
print(df['platform'].unique())

['Wii' 'DS' 'X360' 'PS3' 'PS2' '3DS' 'PS4' 'PS' 'XB' 'PC' 'PSP' 'WiiU'
 'GC' 'GBA' 'XOne' 'PSV' 'DC']


Проверим столбец year_of_release на дубликаты:

In [20]:
print(df['year_of_release'].unique())

[2006. 2008. 2009. 2005. 2007. 2010. 2013. 2004. 2002. 2001. 2011. 2012.
 2014. 1997. 1999. 2015. 2016. 2003. 1998. 1996. 2000. 1994. 1992. 1985.
 1988.]


Проверим столбец rating на дубликаты:

In [21]:
print(df['rating'].unique())

['E' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


Проверим наличие "неявных" дубликатов во всех столбцах:

In [22]:
df.duplicated(subset=['genre', 'name', 'platform', 'year_of_release', 'rating'], keep = False).sum()

268

При подсчете неявных дубликатов в ключевых столбцах видим, что их содержится 268 в общей сумме


In [23]:
df.drop_duplicates(subset=['genre', 'name', 'platform', 'year_of_release', 'rating'], keep = False)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
2,Mario Kart Wii,Wii,2008.0,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
6,New Super Mario Bros.,DS,2006.0,platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,misc,13.96,9.18,2.93,2.84,58.0,6.6,E
...,...,...,...,...,...,...,...,...,...,...,...
16937,Bust-A-Move 3000,GC,2003.0,puzzle,0.01,0.00,0.00,0.00,53.0,NaN,E
16938,Mega Brain Boost,DS,2008.0,puzzle,0.01,0.00,0.00,0.00,48.0,NaN,E
16943,STORM: Frontline Nation,PC,2011.0,strategy,0.00,0.01,0.00,0.00,60.0,7.2,E10+
16945,Plushees,DS,2008.0,simulation,0.01,0.00,0.00,0.00,NaN,NaN,E


- После того как нормализуете данные и устраните неявные дубликаты, проверьте наличие явных дубликатов в данных.

In [24]:
df.duplicated(subset=None)

0        False
2        False
3        False
6        False
7        False
         ...  
16937    False
16938    False
16943    False
16945    False
16947    False
Length: 9892, dtype: bool

Исходя из результата False делаем вывод, что явных дубликатов в строках не обнаружено

Количество явных дубликатов было небольшим и содержалось в основном в столбце с жанром из-за разного написания регистра. Все значения были приведены к нижнему регистру

In [25]:
na_per_column = df.isna().sum()
total_rows = len(df)
relative_na = (df.isna().sum() / total_rows) * 100
print(na_per_column, relative_na)

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       1896
user_score         2425
rating                0
dtype: int64 name                0.000000
platform            0.000000
year_of_release     0.000000
genre               0.000000
na_sales            0.000000
eu_sales            0.000000
jp_sales            0.000000
other_sales         0.000000
critic_score       19.167004
user_score         24.514759
rating              0.000000
dtype: float64


Так как пропуски все равно остались в двух столбцах, то удалим их за невозможностью заменить на усредненные значения:

In [26]:
df = df.dropna(subset = ['critic_score', 'user_score'])

In [27]:
na_per_column = df.isna().sum()
total_rows = len(df)
relative_na = (df.isna().sum() / total_rows) * 100
print(na_per_column, relative_na)

name               0
platform           0
year_of_release    0
genre              0
na_sales           0
eu_sales           0
jp_sales           0
other_sales        0
critic_score       0
user_score         0
rating             0
dtype: int64 name               0.0
platform           0.0
year_of_release    0.0
genre              0.0
na_sales           0.0
eu_sales           0.0
jp_sales           0.0
other_sales        0.0
critic_score       0.0
user_score         0.0
rating             0.0
dtype: float64


После проведения предобработки данных были удалены как строки с многочисленными пропусками по причине невозможности их замены на усредненные значения, так как разброс довольно большой, что приведет к искажению данных, так и строки с небольшим количеством пропусков, так как их количество было мало и на дальнейший анализ это не повлияет.

Также проверили наличие явных и неявных дубликатов: явные дубликаты не были обнаружены, а неявные были обработаны и приведены к общему виду написания

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберем данные по этому показателю. Сохраним новый срез данных в отдельном датафрейме, `df_actual`.

In [28]:
df_actual = df.loc[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]

In [29]:
print(df_actual.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6145 entries, 0 to 16943
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             6145 non-null   object 
 1   platform         6145 non-null   object 
 2   year_of_release  6145 non-null   float64
 3   genre            6145 non-null   object 
 4   na_sales         6145 non-null   float64
 5   eu_sales         6145 non-null   float64
 6   jp_sales         6145 non-null   float64
 7   other_sales      6145 non-null   float64
 8   critic_score     6145 non-null   float64
 9   user_score       6145 non-null   float64
 10  rating           6145 non-null   object 
dtypes: float64(7), object(4)
memory usage: 576.1+ KB
None


---

## 4. Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [30]:
df.loc[:, 'category'] = pd.cut(df['user_score'], bins=[0, 3, 8, 10], labels=['Низкая оценка', 'Средняя оценка', 'Высокая оценка'], right = False)

In [31]:
print(df['category'])

0        Высокая оценка
2        Высокая оценка
3        Высокая оценка
6        Высокая оценка
7        Средняя оценка
              ...      
16902     Низкая оценка
16913    Высокая оценка
16932    Средняя оценка
16936    Средняя оценка
16943    Средняя оценка
Name: category, Length: 6903, dtype: category
Categories (3, object): ['Низкая оценка' < 'Средняя оценка' < 'Высокая оценка']


- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [32]:
df.loc[:, 'category_critic']  = pd.cut(df['critic_score'], bins=[0, 30, 80, 100], labels=['Низкая оценка', 'Средняя оценка', 'Высокая оценка'], right = False)

In [33]:
print(df['category_critic'])

0        Средняя оценка
2        Высокая оценка
3        Высокая оценка
6        Высокая оценка
7        Средняя оценка
              ...      
16902    Средняя оценка
16913    Высокая оценка
16932    Высокая оценка
16936    Средняя оценка
16943    Средняя оценка
Name: category_critic, Length: 6903, dtype: category
Categories (3, object): ['Низкая оценка' < 'Средняя оценка' < 'Высокая оценка']


- После категоризации данных проверим результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [34]:
category_group = df.groupby('category')['user_score'].count()

In [35]:
print(category_group)

category
Низкая оценка      107
Средняя оценка    4396
Высокая оценка    2400
Name: user_score, dtype: int64


In [36]:
category_group_critic = df.groupby('category_critic')['critic_score'].count()

In [37]:
print(category_group_critic)

category_critic
Низкая оценка       52
Средняя оценка    4913
Высокая оценка    1938
Name: critic_score, dtype: int64


- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [38]:
top_platforms = df.groupby('platform')['name'].count().sort_values(ascending=False).head(7)

In [39]:
print(top_platforms)

platform
PS2     1153
X360     866
PS3      779
PC       662
XB       575
Wii      484
DS       468
Name: name, dtype: int64


---

## 5. Итоговый вывод

В завершение работы хочется подчеркнуть, что изначальные данные были предобработаны и было удалено более 5000 строк с пропусками за невозможностью их заменя на усредненное значение, также были выявлены неявные дубликаты, которые были устранены посредством приведения строк столбца к нижнему регистру. Также был создан новый датафрейм только с определенным диапазоном времени (2000-2013 гг) с количеством строк более 6000.

Отдельно были категоризированы данные по оценке пользователей и по оценке критиков, в связи с чем были разделены на 3 группы (низкая, средняя и высокая оценка соответственно). В следствие этого в изначальный датафрейм были добавлены новые столбцы с категоризацией 'category' и 'category_critic'.

Также были выделены топ-7 платформ по количеству игр: видим, что лидирующие позиции занимают такие плотформы как PS2, X360, PS3. Скорее всего эти платформы наиболее востребованы в пользовании у игроков и на них стоит сосредоточить свое внимание

Для выявления "неявных" дубликатов были выделены ключевые для анализа столбцы для проверки: genre, name, platform, year_of_release, rating. С помощью метода duplicated были проверены все ключевые столбцы на все повторы, в результате анализа сумма дупликатов по всем ключевым столбцам составила 268.